In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Validando os lotes dos embeddings gerados pelo Sentence Transformers

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from tqdm.auto import tqdm


EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings"
)

EXPECTED_DIMENSION = 384

embedding_files = sorted(
    EMBEDDINGS_DIR.glob("*.parquet")
)

print("Total de lotes encontrados:", len(embedding_files))


issues = []
total_embeddings = 0


for file_path in tqdm(embedding_files, desc="Validando lotes"):

    try:

        df = pd.read_parquet(file_path)

        if "embedding" not in df.columns:

            issues.append(
                (file_path.name, "coluna embedding ausente")
            )

            continue

        embeddings = np.vstack(
            df["embedding"].values
        )

        total_embeddings += len(embeddings)

        if embeddings.shape[1] != EXPECTED_DIMENSION:

            issues.append(
                (
                    file_path.name,
                    f"dimensão encontrada {embeddings.shape}"
                )
            )

        if embeddings.dtype != np.float32:

            issues.append(
                (
                    file_path.name,
                    f"dtype encontrado {embeddings.dtype}"
                )
            )

        if not np.isfinite(embeddings).all():

            issues.append(
                (
                    file_path.name,
                    "possui NaN ou infinito"
                )
            )

    except Exception as error:

        issues.append(
            (file_path.name, str(error))
        )


print("\n========================")
print("Total embeddings verificados:", total_embeddings)
print("Problemas encontrados:", len(issues))


if issues:

    print("\nProblemas encontrados:")

    for issue in issues[:10]:
        print(issue)

else:

    print("Todas as embeddings estão válidas!")

Verificação de normalização

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

from tqdm.auto import tqdm


EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings"
)


embedding_files = sorted(
    EMBEDDINGS_DIR.glob("*.parquet")
)


embedding_norms = []

print("Total de lotes:", len(embedding_files))


for file_path in tqdm(embedding_files, desc="Verificando normalização"):

    df = pd.read_parquet(file_path)

    embeddings = np.vstack(
        df["embedding"].values
    ).astype(np.float32)

    batch_norms = np.linalg.norm(
        embeddings,
        axis=1
    )

    embedding_norms.extend(batch_norms)


embedding_norms = np.array(
    embedding_norms
)


print("\n========================")
print("Total embeddings analisados:", len(embedding_norms))

print("\nNorma mínima:", embedding_norms.min())
print("Norma máxima:", embedding_norms.max())
print("Norma média:", embedding_norms.mean())

print("\nDesvio padrão:", embedding_norms.std())

Instalação do FAISS

In [ ]:
%pip install faiss-cpu

Criação do índice FAISS

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import faiss
import json
import gc

from tqdm.auto import tqdm


EMBEDDINGS_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/embeddings"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/TCC_Chatbot/FAISS"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DIMENSION = 384

INDEX_PATH = OUTPUT_DIR / "indice_faiss.index"
METADATA_PATH = OUTPUT_DIR / "metadados.json"


embedding_files = sorted(
    EMBEDDINGS_DIR.glob("*.parquet")
)

if not embedding_files:

    raise FileNotFoundError(
        f"Nenhum arquivo .parquet encontrado em:\n{EMBEDDINGS_DIR}"
    )


print("=" * 80)
print("CRIAÇÃO DO ÍNDICE FAISS")
print("=" * 80)

print(f"Arquivos encontrados: {len(embedding_files)}")
print(f"Dimensão esperada: {DIMENSION}")
print()


index = faiss.IndexFlatIP(DIMENSION)

total_vectors = 0

print("Adicionando embeddings ao FAISS...")


for file_path in tqdm(embedding_files, desc="Processando embeddings"):

    df = pd.read_parquet(
        file_path,
        columns=["embedding"]
    )

    if "embedding" not in df.columns:

        raise ValueError(
            f"O arquivo {file_path.name} não possui a coluna 'embedding'."
        )

    if len(df) == 0:

        print(f"\nAviso: {file_path.name} está vazio.")

        continue

    embeddings = np.vstack(
        df["embedding"].values
    ).astype(np.float32)

    if embeddings.ndim != 2:

        raise ValueError(
            f"Formato inválido em {file_path.name}: "
            f"{embeddings.shape}"
        )

    if embeddings.shape[1] != DIMENSION:

        raise ValueError(
            f"Dimensão inválida em {file_path.name}: "
            f"{embeddings.shape[1]} "
            f"(esperado: {DIMENSION})"
        )

    if not np.isfinite(embeddings).all():

        raise ValueError(
            f"Foram encontrados NaN ou infinito em "
            f"{file_path.name}"
        )

    embedding_norms = np.linalg.norm(
        embeddings,
        axis=1
    )

    if not np.allclose(embedding_norms, 1.0, atol=1e-3):

        raise ValueError(
            f"Embeddings não normalizadas em "
            f"{file_path.name}. "
            f"Norma mínima: {embedding_norms.min():.6f}, "
            f"máxima: {embedding_norms.max():.6f}"
        )

    index.add(
        embeddings
    )

    total_vectors += embeddings.shape[0]

    del df
    del embeddings
    del embedding_norms

    gc.collect()


print()
print("=" * 80)
print("RESULTADO")
print("=" * 80)

print(f"Vetores adicionados: {total_vectors:,}")
print(f"Dimensão: {index.d}")
print(f"Total no índice: {index.ntotal:,}")


faiss.write_index(
    index,
    str(INDEX_PATH)
)


metadata = {
    "total_vetores": int(total_vectors),
    "dimensao": int(DIMENSION),
    "tipo": "IndexFlatIP",
    "normalizado": True,
    "modelo": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
}


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


print()
print("=" * 80)
print("ÍNDICE FAISS CRIADO COM SUCESSO")
print("=" * 80)

print("Índice:")
print(INDEX_PATH)

print()
print("Metadados:")
print(METADATA_PATH)

Verificação do indice_faiss

In [ ]:
import faiss


INDEX_PATH = (
    "/content/drive/MyDrive/TCC_Chatbot/FAISS/indice_faiss.index"
)

index = faiss.read_index(INDEX_PATH)

print("Vetores no índice:", index.ntotal)